<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/EDA_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# A purposely messy sales orders dataset.
# Every problem category we cover today is deliberately represented.

raw_orders = pd.DataFrame({
    "order_id": [
        "ORD-001","ORD-002","ORD-003","ORD-004","ORD-005",
        "ORD-006","ORD-007","ORD-002",  # duplicate of ORD-002
        "ORD-008","ORD-009","ORD-010","ORD-011","ORD-012",
        "ORD-013","ORD-001",  # duplicate of ORD-001 (different data — dirty duplicate)
        "ORD-014","ORD-015","ORD-016","ORD-017","ORD-018",
        "ORD-019","ORD-020","ORD-021","ORD-022","ORD-023",
        "ORD-024","ORD-025"
    ],
    "customer_name": [
        "Ananya Sharma","Rohit Mehta","Priya Nair","Vikram Rao","Sneha Pillai",
        "Arjun Das","Deepa Joshi","Rohit Mehta",
        "Kiran Iyer","Meera Bose","Suresh Kumar",None,"Rahul Singh",
        "Divya Nair","Ananya Sharma",
        "Arun Sharma","Pooja Mehta","Sanjay Rao",None,"Tarun Das",
        "Farida Khan","Gaurav Tiwari","Hema Iyer","Ibrahim Sheikh","Jyoti Bhatt",
        "Kabir Anand","Lavanya Menon"
    ],
    "product_category": [
        "Electronics","Clothing","Electronics","Home","Clothing",
        "Electronics","Home","Clothing",
        "Electronics","Clothing",None,"Home","Electronics",
        "clothing",  # wrong casing
        "Electronics",
        "Home","Clothing","Electronics","Home","Clothing",
        "ELECTRONICS",  # wrong casing
        "Home","Clothing","Electronics","Home",
        "Electroncis",  # typo
        "Clothing"
    ],
    "quantity": [
        2, 1, 5, 3, 2,
        4, 1, 1,
        3, 2, 1, None, 6,
        2, 2,
        1, 3, 2, 4, 1,
        2, -1,  # negative quantity — impossible
        3, 1, 500,  # outlier: 500 units
        2, 1
    ],
    "unit_price": [
        15000, 2500, 12000, 8000, 3500,
        18000, 5000, 2500,
        22000, 1800, 6500, 9000, 14000,
        1500, 15000,
        7000, 3200, 11000, 4500, 2800,
        16000, 3100,
        4200, 19000, 13500,
        6800, 2100
    ],
    "total_amount": [
        30000, 2500, 60000, 24000, 7000,
        72000, 5000, 2500,
        66000, 3600, 6500, None, 84000,
        3000, 30000,
        7000, 9600, 22000, 18000, 2800,
        32000, -3100,  # inconsistent: negative total from negative quantity
        12600, 19000, 6750000,  # outlier matching the qty=500 order
        13600, 2100
    ],
    "order_date": [
        "2024-01-05","2024-01-08","2024-01-10","2024-01-15","2024-01-20",
        "2024-02-01","2024-02-05","2024-01-08",
        "2024-02-10","2024-02-14","2024-02-20","2024-03-01","2024-03-05",
        "2024-03-10","2024-01-05",
        "2024-03-15","2024-03-20","2024-04-01","2024-04-05","2024-04-10",
        "2024-04-15","2024-04-20","2024-05-01","2024-05-05","2024-05-10",
        "2024-05-15","2024-05-20"
    ],
    "city": [
        "Mumbai","Delhi","Bangalore","Chennai","Mumbai",
        "Hyderabad","Pune","Delhi",
        "Bangalore","Chennai","Mumbai","Delhi","Hyderabad",
        "Pune","Mumbai",
        "Bangalore","Chennai","Mumbai",None,"Delhi",
        "Hyderabad","Pune","Bangalore","Chennai","Mumbai",
        "Delhi","Hyderabad"
    ],
    "status": [
        "Delivered","Delivered","Shipped","Delivered","Processing",
        "Delivered","Shipped","Delivered",
        "Processing","Delivered","Shipped",None,"Delivered",
        "processing",  # wrong casing
        "Delivered",
        "Shipped","Delivered","Processing","Delivered","Shipped",
        "DELIVERED",  # wrong casing
        "Processing","Delivered","Shipped","Delivered",
        "Deliverd",  # typo
        "Processing"
    ]
})

# Make a frozen copy — we will always compare against this
original = raw_orders.copy()

print(f"Raw dataset shape: {raw_orders.shape}")
print()
print(raw_orders.head(10))

Raw dataset shape: (27, 9)

  order_id  customer_name product_category  quantity  unit_price  \
0  ORD-001  Ananya Sharma      Electronics       2.0       15000   
1  ORD-002    Rohit Mehta         Clothing       1.0        2500   
2  ORD-003     Priya Nair      Electronics       5.0       12000   
3  ORD-004     Vikram Rao             Home       3.0        8000   
4  ORD-005   Sneha Pillai         Clothing       2.0        3500   
5  ORD-006      Arjun Das      Electronics       4.0       18000   
6  ORD-007    Deepa Joshi             Home       1.0        5000   
7  ORD-002    Rohit Mehta         Clothing       1.0        2500   
8  ORD-008     Kiran Iyer      Electronics       3.0       22000   
9  ORD-009     Meera Bose         Clothing       2.0        1800   

   total_amount  order_date       city      status  
0       30000.0  2024-01-05     Mumbai   Delivered  
1        2500.0  2024-01-08      Delhi   Delivered  
2       60000.0  2024-01-10  Bangalore     Shipped  
3       240

---
# Part 1 — The Cleaning Mindset: Audit First, Fix Second

## The most important rule in data cleaning

**Never modify data before you understand what is wrong with it.**

The instinct when you see messy data is to start fixing things immediately. This leads to:
- Fixing the wrong things
- Fixing things in the wrong order (some fixes depend on others)
- Not knowing what you changed or why
- No way to reproduce the cleaning on a future data export

The professional approach is a two-phase workflow:

```
PHASE 1 — AUDIT    Understand everything that is wrong.
                   Document it. Decide what to do about each issue.
                   Do NOT modify data yet.

PHASE 2 — CLEAN   Apply each fix in a logical order.
                   Record what you did and why.
                   Validate that each fix worked.
```

Let's start with the audit.

In [ ]:
# AUDIT STEP 1: Basic shape and types
print("=" * 50)
print("AUDIT REPORT — Sales Orders Dataset")
print("=" * 50)
print(f"Rows:    {raw_orders.shape[0]}")
print(f"Columns: {raw_orders.shape[1]}")
print()
print("Column types:")
print(raw_orders.dtypes)
print()
print("Memory usage:", raw_orders.memory_usage(deep=True).sum() // 1024, "KB")

AUDIT REPORT — Sales Orders Dataset
Rows:    27
Columns: 9

Column types:
order_id             object
customer_name        object
product_category     object
quantity            float64
unit_price            int64
total_amount        float64
order_date           object
city                 object
status               object
dtype: object

Memory usage: 9 KB


In [ ]:
# AUDIT STEP 2: Missing values

missing = raw_orders.isnull().sum()
missing_pct = (missing / raw_orders.shape[0] * 100).round(1)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct":   missing_pct
})
missing_report = missing_report[missing_report["missing_count"] > 0] #Boolean Indexing

print("Missing values:")
print(missing_report)
print(f"\nTotal missing cells: {missing.sum()}")

Missing values:
                  missing_count  missing_pct
customer_name                 2          7.4
product_category              1          3.7
quantity                      1          3.7
total_amount                  1          3.7
city                          1          3.7
status                        1          3.7

Total missing cells: 7


In [ ]:
# AUDIT STEP 3: Duplicates

# Exact row duplicates (every column matches)
exact_dups = raw_orders.duplicated().sum()
print(f"Exact duplicate rows: {exact_dups}")
print()

# Key duplicates — same order_id, even if other columns differ
key_dups = raw_orders.duplicated(subset=["order_id"]).sum()
print(f"Duplicate order_ids: {key_dups}")
print()
print("Which order_ids are duplicated:")
dup_ids = raw_orders[raw_orders.duplicated(subset=["order_id"], keep=False)]
print(dup_ids.sort_values("order_id")[["order_id", "customer_name", "total_amount", "order_date"]])

Exact duplicate rows: 2

Duplicate order_ids: 2

Which order_ids are duplicated:
   order_id  customer_name  total_amount  order_date
0   ORD-001  Ananya Sharma       30000.0  2024-01-05
14  ORD-001  Ananya Sharma       30000.0  2024-01-05
1   ORD-002    Rohit Mehta        2500.0  2024-01-08
7   ORD-002    Rohit Mehta        2500.0  2024-01-08


In [ ]:
# AUDIT STEP 4: Categorical column values — spot inconsistencies

for col in ["product_category", "status"]:
    print(f"{col} unique values:")
    print(raw_orders[col].value_counts(dropna=True))
    print()

product_category unique values:
product_category
Electronics    8
Clothing       8
Home           7
clothing       1
ELECTRONICS    1
Electroncis    1
Name: count, dtype: int64

status unique values:
status
Delivered     12
Shipped        6
Processing     5
processing     1
DELIVERED      1
Deliverd       1
Name: count, dtype: int64



In [ ]:
# AUDIT STEP 5: Numeric columns — look for impossible values and outliers

print("Numeric summary:")
print(raw_orders[["quantity", "unit_price", "total_amount"]].describe().round(0))
print()

print("Suspicious values:")
print("  Negative quantities:", (raw_orders["quantity"] < 0).sum())
print("  Negative totals:    ", (raw_orders["total_amount"] < 0).sum())
print("  Quantity > 100:     ", (raw_orders["quantity"] > 100).sum())
print()

print("Rows with negative or extreme values:")
suspect = raw_orders[
    (raw_orders["quantity"] < 0) |
    (raw_orders["total_amount"] < 0) |
    (raw_orders["quantity"] > 100)
]
print(suspect[["order_id", "quantity", "unit_price", "total_amount"]])

Numeric summary:
       quantity  unit_price  total_amount
count      26.0        27.0          26.0
mean       21.0      8500.0      280065.0
std        98.0      6124.0     1319827.0
min        -1.0      1500.0       -3100.0
25%         1.0      3150.0        3950.0
50%         2.0      6800.0       13100.0
75%         3.0     13750.0       30000.0
max       500.0     22000.0     6750000.0

Suspicious values:
  Negative quantities: 1
  Negative totals:     1
  Quantity > 100:      1

Rows with negative or extreme values:
   order_id  quantity  unit_price  total_amount
21  ORD-020      -1.0        3100       -3100.0
24  ORD-023     500.0       13500     6750000.0


### Whiteboard: audit findings summary

Before touching a single value, write down every problem and the planned fix:

| # | Problem | Columns affected | Planned fix |
|---|---|---|---|
| 1 | Missing values | `customer_name`, `product_category`, `quantity`, `total_amount`, `city`, `status` | Fill or drop depending on column |
| 2 | Exact duplicate rows | All | Drop with `drop_duplicates()` |
| 3 | Key duplicates (same order_id, different data) | `order_id` | Investigate — keep most recent |
| 4 | Wrong casing in categoricals | `product_category`, `status` | `.str.strip().str.title()` |
| 5 | Typos in categoricals | `product_category` (`Electroncis`), `status` (`Deliverd`) | `.replace()` |
| 6 | Negative quantity / total | `quantity`, `total_amount` | Flag and remove |
| 7 | Extreme outlier (qty=500) | `quantity`, `total_amount` | Investigate — likely data entry error |
| 8 | `total_amount` ≠ `qty × price` | `total_amount` | Recompute from trusted columns |
| 9 | `order_date` as string | `order_date` | Convert to datetime64 |

---
# Part 2 — Detecting and Filling Missing Values

## The decision tree for missing values

Missing values are not all equal. Before filling or dropping, ask:

```
WHY is this value missing?

  Random / data entry error  ->  Fill with a sensible substitute
  Required field / no default ->  Drop the row
  A large % of rows missing   ->  Consider dropping the COLUMN
  Informative absence         ->  Keep as NaN or create an 'is_missing' flag
```

**Filling strategies:**

| Strategy | Use when | Pandas method |
|---|---|---|
| Fill with a fixed value | Default makes sense (e.g. 0 for a count) | `.fillna(value)` |
| Fill with column mean/median | Numeric column, symmetric or skewed distribution | `.fillna(col.mean())` |
| Forward fill | Time-series where previous value carries forward | `.fillna(method='ffill')` |
| Fill with mode | Categorical column (most common value) | `.fillna(col.mode()[0])` |
| Drop the row | Required field, no sensible default | `.dropna(subset=['col'])` |

In [ ]:
# Work on a copy — never modify the original during cleaning
orders = raw_orders.copy()

print("Missing values BEFORE filling:")
print(orders.isnull().sum()[orders.isnull().sum() > 0])
print()

Missing values BEFORE filling:
customer_name       2
product_category    1
quantity            1
total_amount        1
city                1
status              1
dtype: int64



In [ ]:
# quantity: numeric, only 1 missing — fill with the median
# (median is more robust than mean when there is an outlier like qty=500)

qty_median = orders["quantity"].median()
print(f"Filling missing quantity with median: {qty_median}")
orders["quantity"] = orders["quantity"].fillna(qty_median)

# total_amount: 1 missing — will be recomputed from qty * price later anyway
# We flag it now with a sentinel and recompute after other fixes are done
orders["total_needs_recompute"] = orders["total_amount"].isnull()
print(f"total_amount rows flagged for recompute: {orders['total_needs_recompute'].sum()}")

Filling missing quantity with median: 2.0
total_amount rows flagged for recompute: 1


In [ ]:
orders

,order_id,customer_name,product_category,quantity,unit_price,total_amount,order_date,city,status,total_needs_recompute
0,ORD-001,Ananya Sharma,Electronics,2.0,15000,30000.0,2024-01-05,Mumbai,Delivered,False
1,ORD-002,Rohit Mehta,Clothing,1.0,2500,2500.0,2024-01-08,Delhi,Delivered,False
2,ORD-003,Priya Nair,Electronics,5.0,12000,60000.0,2024-01-10,Bangalore,Shipped,False
3,ORD-004,Vikram Rao,Home,3.0,8000,24000.0,2024-01-15,Chennai,Delivered,False
4,ORD-005,Sneha Pillai,Clothing,2.0,3500,7000.0,2024-01-20,Mumbai,Processing,False
5,ORD-006,Arjun Das,Electronics,4.0,18000,72000.0,2024-02-01,Hyderabad,Delivered,False
6,ORD-007,Deepa Joshi,Home,1.0,5000,5000.0,2024-02-05,Pune,Shipped,False
7,ORD-002,Rohit Mehta,Clothing,1.0,2500,2500.0,2024-01-08,Delhi,Delivered,False
8,ORD-008,Kiran Iyer,Electronics,3.0,22000,66000.0,2024-02-10,Bangalore,Processing,False
9,ORD-009,Meera Bose,Clothing,2.0,1800,3600.0,2024-02-14,Chennai,Delivered,False


In [ ]:
# product_category: fill with mode (most common category)
cat_mode = orders["product_category"].mode()[0]
print(f"Most common product_category: '{cat_mode}'  — using as fill value")
orders["product_category"] = orders["product_category"].fillna(cat_mode)

# status: fill with 'Unknown' — we cannot infer the status
orders["status"] = orders["status"].fillna("Unknown")

print()
print("product_category after fill:", orders["product_category"].isnull().sum(), "missing")
print("status after fill:           ", orders["status"].isnull().sum(), "missing")

Most common product_category: 'Clothing'  — using as fill value

product_category after fill: 0 missing
status after fill:            0 missing


In [ ]:
# customer_name and city: required for a meaningful order record
# No sensible fill — drop rows where either is missing

rows_before = len(orders)
orders = orders.dropna(subset=["customer_name", "city"])
rows_after = len(orders)

print(f"Rows dropped (missing customer_name or city): {rows_before - rows_after}")
print(f"Rows remaining: {rows_after}")
print()
print("Missing values AFTER all fills and drops:")
remaining_missing = orders.isnull().sum()
print(remaining_missing[remaining_missing > 0])
if remaining_missing.sum() == 0:
    print("No missing values remain (except the recompute flag column).")

Rows dropped (missing customer_name or city): 2
Rows remaining: 25

Missing values AFTER all fills and drops:
Series([], dtype: int64)
No missing values remain (except the recompute flag column).


---
# Part 3 — Removing Duplicates

## Two types of duplicates — and why they need different treatment

**Type 1: Exact duplicates** — every column in two rows is identical. These are always safe to drop — they are simply the same record entered twice.

**Type 2: Key duplicates** — the primary key (e.g. `order_id`) appears more than once, but other columns differ. This is more dangerous — it usually means two systems recorded the same order differently. Simply dropping one row might discard the correct version. You need to decide which copy to keep.

**Strategies for key duplicates:**
- Keep the **first** occurrence (trust the original entry)
- Keep the **last** occurrence (trust the most recent update)
- Keep the row where a specific column has the highest/most complete value
- Flag both rows and investigate manually

In [ ]:
# Type 1: exact row duplicates

print(f"Rows before dropping exact duplicates: {len(orders)}")

# keep='first' means: keep the first occurrence, mark subsequent ones as duplicate
orders = orders.drop_duplicates(keep="first")

print(f"Rows after dropping exact duplicates:  {len(orders)}")
print(f"Exact duplicates removed: {len(raw_orders) - len(orders)} (net so far)")

Rows before dropping exact duplicates: 25
Rows after dropping exact duplicates:  23
Exact duplicates removed: 4 (net so far)


In [ ]:
# Type 2: key duplicates — same order_id, different data

key_dups_remaining = raw_orders.duplicated(subset=["order_id"], keep=False)
print(f"Rows with a duplicated order_id: {key_dups_remaining.sum()}")
print()
print("The duplicated orders:")
print(raw_orders[key_dups_remaining].sort_values("order_id")
      [["order_id", "customer_name", "total_amount", "order_date"]])
print()
print("These have the SAME order_id but different data.")
print("Decision: keep the LAST occurrence — it is more likely to be the corrected version.")

Rows with a duplicated order_id: 4

The duplicated orders:
   order_id  customer_name  total_amount  order_date
0   ORD-001  Ananya Sharma       30000.0  2024-01-05
14  ORD-001  Ananya Sharma       30000.0  2024-01-05
1   ORD-002    Rohit Mehta        2500.0  2024-01-08
7   ORD-002    Rohit Mehta        2500.0  2024-01-08

These have the SAME order_id but different data.
Decision: keep the LAST occurrence — it is more likely to be the corrected version.


In [ ]:
# Keep the last occurrence for each order_id

rows_before = len(raw_orders)
orders = raw_orders.drop_duplicates(subset=["order_id"], keep="last")

print(f"Rows after deduplication on order_id: {len(orders)} (was {rows_before})")
print(f"Remaining order_id duplicates: {orders.duplicated(subset=['order_id']).sum()}")
print()

# Reset the index after dropping rows so it is clean and sequential
orders = orders.reset_index(drop=False)
print(f"Index reset. New index range: 0 to {len(orders)-1}")

Rows after deduplication on order_id: 25 (was 27)
Remaining order_id duplicates: 0

Index reset. New index range: 0 to 24


---
# Part 4 — Outlier Detection

## What is an outlier?

An **outlier** is a value that is far from the rest of the data. Outliers are not always errors — sometimes they are genuine extreme events (a very large corporate order, for example). But they are always worth investigating.

**Two standard detection methods:**

**1. IQR method (Interquartile Range)** — robust, works well for skewed data
```
Q1 = 25th percentile
Q3 = 75th percentile
IQR = Q3 - Q1

Lower fence = Q1 - 1.5 * IQR
Upper fence = Q3 + 1.5 * IQR

Anything outside the fences is a potential outlier.
```

**2. Z-score method** — measures how many standard deviations from the mean
```
z = (value - mean) / std

|z| > 3  is a common threshold for flagging outliers
```

In [ ]:
# IQR method on quantity

Q1=orders["quantity"].quantile(0.25)
Q3=orders["quantity"].quantile(0.75)
IQR=Q3-Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"quantity statistics:")
print(f"  Q1 (25th pct):  {Q1}")
print(f"  Q3 (75th pct):  {Q3}")
print(f"  IQR:            {IQR}")
print(f"  Lower fence:    {lower_fence}")
print(f"  Upper fence:    {upper_fence}")
print()

qty_outliers=orders[(orders["quantity"]<lower_fence) | (orders["quantity"]>upper_fence)]

print(f"Outliers by IQR method: {len(qty_outliers)}")
print(qty_outliers[["order_id", "customer_name", "quantity", "unit_price", "total_amount"]])

quantity statistics:
  Q1 (25th pct):  1.0
  Q3 (75th pct):  3.0
  IQR:            2.0
  Lower fence:    -2.0
  Upper fence:    6.0

Outliers by IQR method: 1
   order_id customer_name  quantity  unit_price  total_amount
22  ORD-023   Jyoti Bhatt     500.0       13500     6750000.0


In [ ]:
# Z-score method on total_amount

mean_total = orders["total_amount"].mean()
std_total  = orders["total_amount"].std()

orders["total_zscore"] = (orders["total_amount"] - mean_total) / std_total

zscore_outliers = orders[orders["total_zscore"].abs() > 3]

print(f"total_amount mean: {mean_total:,.0f}")
print(f"total_amount std:  {std_total:,.0f}")
print()
print(f"Outliers by Z-score (|z| > 3): {len(zscore_outliers)}")
print(zscore_outliers[["order_id", "quantity", "unit_price", "total_amount", "total_zscore"]])

total_amount mean: 302,050
total_amount std:  1,373,624

Outliers by Z-score (|z| > 3): 1
   order_id  quantity  unit_price  total_amount  total_zscore
22  ORD-023     500.0       13500     6750000.0      4.694115


In [ ]:
# Domain-knowledge outliers: values that are statistically plausible but logically impossible
# These are found by applying business rules, not statistical methods

# Rule 1: quantity must be >= 1
impossible_qty = orders[orders["quantity"] < 1]
print(f"Rows with quantity < 1 (impossible): {len(impossible_qty)}")
print(impossible_qty[["order_id", "quantity", "total_amount"]])
print()

# Rule 2: total_amount must be > 0
impossible_total = orders[orders["total_amount"] < 0]
print(f"Rows with total_amount < 0 (impossible): {len(impossible_total)}")
print(impossible_total[["order_id", "quantity", "unit_price", "total_amount"]])

Rows with quantity < 1 (impossible): 1
   order_id  quantity  total_amount
19  ORD-020      -1.0       -3100.0

Rows with total_amount < 0 (impossible): 1
   order_id  quantity  unit_price  total_amount
19  ORD-020      -1.0        3100       -3100.0


In [ ]:
# Domain-knowledge outliers: values that are statistically plausible but logically impossible
# These are found by applying business rules, not statistical methods

# Rule 1: quantity must be >= 1
impossible_qty = orders[orders["quantity"] < 1]
print(f"Rows with quantity < 1 (impossible): {len(impossible_qty)}")
print(impossible_qty[["order_id", "quantity", "total_amount"]])
print()

# Rule 2: total_amount must be > 0
impossible_total = orders[orders["total_amount"] < 0]
print(f"Rows with total_amount < 0 (impossible): {len(impossible_total)}")
print(impossible_total[["order_id", "quantity", "unit_price", "total_amount"]])

Rows with quantity < 1 (impossible): 1
   order_id  quantity  total_amount
19  ORD-020      -1.0       -3100.0

Rows with total_amount < 0 (impossible): 1
   order_id  quantity  unit_price  total_amount
19  ORD-020      -1.0        3100       -3100.0


---
# Part 5 — Consistency Checks

## What is a consistency check?

A consistency check verifies that values **agree with each other** or **conform to a known rule**. Missing values and outliers are problems with individual cells. Consistency problems are relationships between cells in the same row — or between a value and a constraint it must satisfy.

Common examples:
- `total_amount` should equal `quantity × unit_price`
- `end_date` should be after `start_date`
- `status == 'Delivered'` implies a delivery date should exist
- A category column should only contain values from a known allowed list
- An `age` column should be between 0 and 120

In [ ]:
# Fix 1: Standardise categorical columns
# product_category and status have wrong casing and typos

print("product_category BEFORE:")
print(orders["product_category"].value_counts())
print()

orders["product_category"] = (
    orders["product_category"]
    .str.strip()
    .str.title()
    .replace({"Electroncis": "Electronics"})  # fix typo
)

print("product_category AFTER:")
print(orders["product_category"].value_counts())
print()

# Validate against allowed list
valid_categories = {"Electronics", "Clothing", "Home"}
invalid_cats = orders[~orders["product_category"].isin(valid_categories)]
print(f"Rows with invalid product_category: {len(invalid_cats)}")

product_category BEFORE:
product_category
Electronics    7
Home           7
Clothing       7
clothing       1
ELECTRONICS    1
Electroncis    1
Name: count, dtype: int64

product_category AFTER:
product_category
Electronics    9
Clothing       8
Home           7
Name: count, dtype: int64

Rows with invalid product_category: 1


In [ ]:
# Fix status column the same way

print("status BEFORE:")
print(orders["status"].value_counts())
print()

orders["status"] = (
    orders["status"]
    .str.strip()
    .str.title()
    .replace({"Deliverd": "Delivered"})  # fix typo
)

print("status AFTER:")
print(orders["status"].value_counts())
print()

valid_statuses = {"Processing", "Shipped", "Delivered", "Unknown"}
invalid_statuses = orders[~orders["status"].isin(valid_statuses)]
print(f"Rows with invalid status: {len(invalid_statuses)}")

status BEFORE:
status
Delivered     10
Shipped        6
Processing     5
processing     1
DELIVERED      1
Deliverd       1
Name: count, dtype: int64

status AFTER:
status
Delivered     12
Shipped        6
Processing     6
Name: count, dtype: int64

Rows with invalid status: 1


In [ ]:
# Fix 2: Recompute total_amount from quantity × unit_price
# This resolves: missing totals, negative totals from bad quantities,
# and inconsistencies where total_amount does not match qty × price

orders["total_amount"] = orders["quantity"] * orders["unit_price"]

# Drop the helper column we no longer need
orders = orders.drop(columns=["total_needs_recompute"])

print("total_amount recomputed from quantity × unit_price.")
print(orders[["order_id", "quantity", "unit_price", "total_amount"]].head(8))

total_amount recomputed from quantity × unit_price.
  order_id  quantity  unit_price  total_amount
0  ORD-003       5.0       12000       60000.0
1  ORD-004       3.0        8000       24000.0
2  ORD-005       2.0        3500        7000.0
3  ORD-006       4.0       18000       72000.0
4  ORD-007       1.0        5000        5000.0
5  ORD-002       1.0        2500        2500.0
6  ORD-008       3.0       22000       66000.0
7  ORD-009       2.0        1800        3600.0


In [ ]:
# Fix 3: Parse order_date as datetime

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

# Check if any dates failed to parse
bad_dates = orders["order_date"].isnull().sum()
print(f"Unparseable dates after conversion: {bad_dates}")
print(f"order_date dtype: {orders['order_date'].dtype}")
print()
print("Date range:")
print(f"  Earliest: {orders['order_date'].min().date()}")
print(f"  Latest:   {orders['order_date'].max().date()}")

Unparseable dates after conversion: 0
order_date dtype: datetime64[ns]

Date range:
  Earliest: 2024-01-05
  Latest:   2024-05-20


---
# Part 6 — The Repeatable Cleaning Pipeline

## Why a function instead of a script

Everything we did above worked on one file. But this company exports a new orders file every month. If we have to repeat each step manually every time, we will:
- Make mistakes
- Forget steps
- Not know what was done to the data

The professional solution: put the entire cleaning workflow into a **function** that accepts a raw DataFrame and returns a clean one. Run it on any export with one call. The same steps, in the same order, every time.

This is the core idea behind a **data pipeline**.

In [ ]:
def clean_orders(df_raw):
    """
    Cleans a raw sales orders DataFrame.

    Steps applied (in order):
      1.  Work on a copy — never modify the input
      2.  Fill missing values
      3.  Drop rows with required fields missing
      4.  Drop exact duplicate rows
      5.  Drop key duplicates (keep last)
      6.  Remove rows with impossible quantity (< 1)
      7.  Flag quantity outliers (IQR method)
      8.  Standardise categorical columns
      9.  Recompute total_amount from quantity * unit_price
      10. Parse order_date as datetime
      11. Reset index

    Returns:
      cleaned DataFrame, audit_log dict
    """
    df   = df_raw.copy()
    log  = {}    # records what happened at each step
    log["rows_input"] = len(df)

    # ── Step 2: Fill missing values ───────────────────────────────────────────
    df["quantity"]         = df["quantity"].fillna(df["quantity"].median())
    df["product_category"] = df["product_category"].fillna(df["product_category"].mode()[0])
    df["status"]           = df["status"].fillna("Unknown")
    df["total_needs_recompute"] = df["total_amount"].isnull()

    # ── Step 3: Drop rows missing required fields ─────────────────────────────
    before = len(df)
    df = df.dropna(subset=["customer_name", "city"])
    log["rows_dropped_missing_required"] = before - len(df)

    # ── Step 4: Drop exact duplicates ────────────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(keep="first")
    log["rows_dropped_exact_duplicates"] = before - len(df)

    # ── Step 5: Drop key duplicates ───────────────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=["order_id"], keep="last")
    log["rows_dropped_key_duplicates"] = before - len(df)

    # ── Step 6: Remove impossible quantities ──────────────────────────────────
    before = len(df)
    df = df[df["quantity"] >= 1]
    log["rows_dropped_impossible_qty"] = before - len(df)

    # ── Step 7: Flag quantity outliers (IQR) ──────────────────────────────────
    Q1, Q3  = df["quantity"].quantile(0.25), df["quantity"].quantile(0.75)
    IQR     = Q3 - Q1
    uf      = Q3 + 1.5 * IQR
    df["qty_outlier_flag"] = df["quantity"] > uf
    log["qty_outliers_flagged"] = int(df["qty_outlier_flag"].sum())

    # ── Step 8: Standardise categoricals ─────────────────────────────────────
    df["product_category"] = (
        df["product_category"].str.strip().str.title()
        .replace({"Electroncis": "Electronics"})
    )
    df["status"] = (
        df["status"].str.strip().str.title()
        .replace({"Deliverd": "Delivered"})
    )

    # ── Step 9: Recompute total ────────────────────────────────────────────────
    df["total_amount"] = df["quantity"] * df["unit_price"]
    df = df.drop(columns=["total_needs_recompute"])

    # ── Step 10: Parse dates ──────────────────────────────────────────────────
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    # ── Step 11: Reset index ──────────────────────────────────────────────────
    df = df.reset_index(drop=True)

    log["rows_output"] = len(df)
    log["rows_removed_total"] = log["rows_input"] - log["rows_output"]

    return df, log


# Run the pipeline on the raw data
cleaned, audit_log = clean_orders(raw_orders)

print("CLEANING PIPELINE AUDIT LOG")
print("=" * 45)
for key, value in audit_log.items():
    print(f"  {key:<40} {value}")
print()
print(f"Final cleaned shape: {cleaned.shape}")

CLEANING PIPELINE AUDIT LOG
  rows_input                               27
  rows_dropped_missing_required            2
  rows_dropped_exact_duplicates            2
  rows_dropped_key_duplicates              0
  rows_dropped_impossible_qty              1
  qty_outliers_flagged                     1
  rows_output                              22
  rows_removed_total                       5

Final cleaned shape: (22, 10)


In [ ]:
# Final validation: run all consistency checks on the cleaned output

checks = {
    "No missing values":          cleaned.isnull().sum().sum() == 0,
    "No exact duplicates":        cleaned.duplicated().sum() == 0,
    "No duplicate order_ids":     cleaned.duplicated(subset=["order_id"]).sum() == 0,
    "All quantities >= 1":        (cleaned["quantity"] >= 1).all(),
    "All totals > 0":             (cleaned["total_amount"] > 0).all(),
    "total = qty * price":        ((cleaned["total_amount"] -
                                    cleaned["quantity"] * cleaned["unit_price"]).abs() < 1).all(),
    "Valid categories only":      cleaned["product_category"].isin(
                                    {"Electronics","Clothing","Home"}).all(),
    "Valid statuses only":        cleaned["status"].isin(
                                    {"Processing","Shipped","Delivered","Unknown"}).all(),
    "order_date is datetime":     str(cleaned["order_date"].dtype).startswith("datetime"),
}

print("VALIDATION RESULTS")
print("=" * 45)
all_passed = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_passed = False
    print(f"  [{status}]  {check}")

print()
print(f"All checks passed: {all_passed}")

VALIDATION RESULTS
  [PASS]  No missing values
  [PASS]  No exact duplicates
  [PASS]  No duplicate order_ids
  [PASS]  All quantities >= 1
  [PASS]  All totals > 0
  [PASS]  total = qty * price
  [PASS]  Valid categories only
  [PASS]  Valid statuses only
  [PASS]  order_date is datetime

All checks passed: True


---
## Session Summary

### The cleaning mindset
- **Audit first, fix second** — never modify data before understanding what is wrong
- Document every finding and every decision in a log
- Always work on a `.copy()` — never modify the raw input
- After every fix, validate that it worked

### Missing values
- `df.isnull().sum()` — count missing per column
- `.fillna(value)` — fill with a constant
- `.fillna(col.median())` — fill numeric with median (robust to outliers)
- `.fillna(col.mode()[0])` — fill categorical with most common value
- `.dropna(subset=['col'])` — drop rows where a required column is missing
- Choose strategy based on WHY the value is missing, not just that it is missing

### Duplicates
- `df.duplicated().sum()` — count exact duplicate rows
- `df.duplicated(subset=['key']).sum()` — count key duplicates
- `df.drop_duplicates(keep='first')` — remove exact duplicates
- `df.drop_duplicates(subset=['key'], keep='last')` — remove key duplicates, keep most recent
- Always `reset_index(drop=True)` after dropping rows

### Outlier detection
- **IQR method:** lower = Q1 − 1.5×IQR, upper = Q3 + 1.5×IQR. Robust for skewed data.
- **Z-score method:** `(value − mean) / std`. Flag if `|z| > 3`. Assumes normal distribution.
- **Domain knowledge:** business rules catch impossible values that statistics miss (negative quantities, impossible dates)
- Do not blindly drop outliers — flag them first, investigate, then decide

### Consistency checks
- Derived columns should be recomputed from trusted source columns
- Categorical columns should be validated against an explicit allowed set with `.isin()`
- Date columns should be converted with `pd.to_datetime(errors='coerce')` — unparseable values become NaN
- Cross-column rules (total = qty × price) should be verified after fixing individual columns

### The repeatable pipeline
- Wrap all cleaning steps inside a function that takes raw data and returns clean data + an audit log
- The audit log makes the pipeline transparent — you always know what was changed and how much
- Finish every pipeline with a validation check block that asserts your quality expectations

---

### Quick reference

```python
# Missing values
df.isnull().sum()                   # count per column
df.isnull().sum().sum()             # total count
df['col'].fillna(df['col'].median())  # fill numeric
df['col'].fillna(df['col'].mode()[0]) # fill categorical
df.dropna(subset=['required_col'])  # drop rows

# Duplicates
df.duplicated().sum()               # exact dup count
df.drop_duplicates(keep='first')    # remove exact
df.drop_duplicates(subset=['id'], keep='last')  # key dups
df.reset_index(drop=True)           # clean index after drop

# Outliers — IQR
Q1, Q3 = df['col'].quantile([0.25, 0.75])
IQR    = Q3 - Q1
df[(df['col'] < Q1 - 1.5*IQR) | (df['col'] > Q3 + 1.5*IQR)]

# Outliers — Z-score
z = (df['col'] - df['col'].mean()) / df['col'].std()
df[z.abs() > 3]

# Consistency
df['col'].isin({'A', 'B', 'C'})     # validate allowed values
pd.to_datetime(df['date'], errors='coerce')  # parse dates safely
(df['total'] - df['qty'] * df['price']).abs() > 1  # cross-check
```

# Exploratory Data Analysis (EDA) Concepts

---

## What we will cover today

1. What EDA is — and why it comes before any model or report
2. Univariate analysis — understanding one variable at a time
3. Multivariate analysis — understanding relationships between variables
4. Correlation analysis — measuring how strongly variables move together
5. Identifying trends — spotting patterns across time and categories
6. Forming hypotheses — turning observations into testable statements

---

> **A new chapter:** Module 2 taught us to load, clean, filter, join, and transform data. All of that was preparation. Module 3 is where the actual analysis begins — asking questions of clean data and drawing conclusions. Today's session introduces the mental model behind Exploratory Data Analysis: how a data scientist systematically moves from a clean dataset to a set of well-formed hypotheses. We use a new, richer dataset purpose-built for this kind of exploration.

In [ ]:
import pandas as pd
import numpy as np

# A richer dataset: student performance records across a coaching institute.
# 200 students, multiple numeric scores, several categorical attributes,
# and deliberate patterns baked in for EDA to discover.

np.random.seed(42)
n = 200

# Categorical attributes
gender        = np.random.choice(["Male", "Female"], n, p=[0.52, 0.48])
stream        = np.random.choice(["Science", "Commerce", "Arts"], n, p=[0.45, 0.35, 0.20])
school_type   = np.random.choice(["Private", "Public"], n, p=[0.55, 0.45])
city_tier     = np.random.choice(["Tier 1", "Tier 2", "Tier 3"], n, p=[0.40, 0.35, 0.25])
study_group   = np.random.choice(["Group A", "Group B", "Group C", "Group D"], n)
parent_edu    = np.random.choice(["Primary", "Secondary", "Graduate", "Postgraduate"], n,
                                  p=[0.10, 0.25, 0.40, 0.25])

# Study hours: Private school students study more on average
study_hours   = np.where(
    school_type == "Private",
    np.random.normal(5.5, 1.2, n),
    np.random.normal(4.2, 1.4, n)
).clip(1, 10).round(1)

# Previous score (class 10 marks, out of 100)
prev_score    = np.random.normal(72, 12, n).clip(35, 99).round(1)

# Test scores: influenced by study hours, previous score, and stream
stream_boost  = np.where(stream == "Science", 5, np.where(stream == "Commerce", 2, -3))
base_score    = (prev_score * 0.4 + study_hours * 5 + stream_boost +
                 np.random.normal(0, 8, n))
math_score    = (base_score + np.random.normal(0, 6, n)).clip(30, 100).round(1)
science_score = (base_score + np.random.normal(2, 7, n)).clip(30, 100).round(1)
english_score = (base_score - stream_boost * 0.5 + np.random.normal(0, 9, n)).clip(30, 100).round(1)

# Attendance: correlated with study hours
attendance    = (study_hours * 7 + np.random.normal(0, 8, n)).clip(50, 100).round(1)

# Tuition: Private school students more likely to have tuition
tuition       = np.where(
    school_type == "Private",
    np.random.choice(["Yes", "No"], n, p=[0.70, 0.30]),
    np.random.choice(["Yes", "No"], n, p=[0.35, 0.65])
)

# Final score: average of the three subject scores
final_score   = ((math_score + science_score + english_score) / 3).round(1)

# Grade derived from final score
grade = pd.cut(final_score,
               bins=[0, 50, 60, 70, 80, 101],
               labels=["F", "C", "B", "A", "A+"])

students = pd.DataFrame({
    "student_id":    [f"S{i+1:03d}" for i in range(n)],
    "gender":        gender,
    "stream":        stream,
    "school_type":   school_type,
    "city_tier":     city_tier,
    "parent_edu":    parent_edu,
    "study_group":   study_group,
    "tuition":       tuition,
    "study_hours":   study_hours,
    "attendance":    attendance,
    "prev_score":    prev_score,
    "math_score":    math_score,
    "science_score": science_score,
    "english_score": english_score,
    "final_score":   final_score,
    "grade":         grade
})

print(f"Dataset shape: {students.shape}")
print()
print(students.head(5))
print()
print(students.dtypes)

Dataset shape: (200, 16)

  student_id  gender    stream school_type city_tier parent_edu study_group  \
0       S001    Male  Commerce     Private    Tier 1  Secondary     Group D   
1       S002  Female   Science      Public    Tier 1    Primary     Group C   
2       S003  Female   Science     Private    Tier 1  Secondary     Group B   
3       S004  Female      Arts      Public    Tier 1   Graduate     Group A   
4       S005    Male  Commerce     Private    Tier 1   Graduate     Group D   

  tuition  study_hours  attendance  prev_score  math_score  science_score  \
0      No          7.2        50.0        86.7        79.2           86.2   
1      No          3.3        50.0        57.5        30.0           30.0   
2     Yes          4.5        50.0        92.1        71.4           82.1   
3      No          5.8        50.0        77.0        67.8           60.1   
4      No          4.8        50.0        63.5        55.7           65.7   

   english_score  final_score grade 

---
# Part 1 — What Is EDA and Why Does It Come First?

## The definition

**Exploratory Data Analysis (EDA)** is the process of systematically examining a dataset to:
- Understand what it contains
- Find patterns, relationships, and anomalies
- Form hypotheses worth testing further
- Decide what analysis or model to build next

## The wrong order — and why it is common

Many beginners jump straight from loading data to building a model or writing a report. The typical result:
- The model performs poorly because the data was misunderstood
- The report answers the wrong question
- A relationship that would have been obvious from EDA is missed entirely

## The right order

```
1. Clean the data              (Module 2 — done)
2. Explore the data (EDA)      <- today
3. Form hypotheses             <- today
4. Test the hypotheses         (statistics / models — coming soon)
5. Communicate findings        (visualisation / reporting — next sessions)
```

## The EDA mindset

EDA is not about getting answers — it is about asking better questions.

You are not yet claiming anything. You are looking at the data from every angle, noting what is surprising, and building a list of things worth investigating properly. The output of EDA is a set of hypotheses, not a set of conclusions.

In [ ]:
# The EDA starting point: always the same three cells

print("Shape:", students.shape)
print()
print("Column types:")
print(students.dtypes)
print()
print("Missing values:")
print(students.isnull().sum())

Shape: (200, 16)

Column types:
student_id         object
gender             object
stream             object
school_type        object
city_tier          object
parent_edu         object
study_group        object
tuition            object
study_hours       float64
attendance        float64
prev_score        float64
math_score        float64
science_score     float64
english_score     float64
final_score       float64
grade            category
dtype: object

Missing values:
student_id       0
gender           0
stream           0
school_type      0
city_tier        0
parent_edu       0
study_group      0
tuition          0
study_hours      0
attendance       0
prev_score       0
math_score       0
science_score    0
english_score    0
final_score      0
grade            0
dtype: int64


In [ ]:
# Full numeric summary
print("Numeric summary:")
print(students.describe().round(2))

Numeric summary:
       study_hours  attendance  prev_score  math_score  science_score  \
count       200.00      200.00      200.00      200.00         200.00   
mean          5.01       50.64       73.00       57.37          59.18   
std           1.55        2.34       11.68       13.76          14.07   
min           1.00       50.00       40.90       30.00          30.00   
25%           4.00       50.00       64.70       47.85          49.75   
50%           5.05       50.00       72.95       58.35          61.25   
75%           6.00       50.00       80.53       67.42          69.43   
max           8.50       66.50       99.00       86.10          89.90   

       english_score  final_score  
count         200.00       200.00  
mean           56.29        57.62  
std            14.96        13.10  
min            30.00        30.00  
25%            44.98        48.40  
50%            55.40        58.15  
75%            65.82        67.08  
max            95.60        88.20  


---
# Part 2 — Univariate Analysis

## What it means

**Univariate** = one variable at a time.

Before looking at how variables relate to each other, understand each one in isolation:
- What is the range?
- Where is the centre (mean vs median)?
- How spread out are the values?
- What is the shape of the distribution — symmetric, skewed, bimodal?
- For categorical columns: how many categories, how balanced are they?

The goal is to build a mental model of each column before mixing them together.

## Whiteboard: the two types of columns

```
NUMERIC columns          CATEGORICAL columns
-----------------        --------------------
mean, median, std        value_counts()
min, max, range          nunique()
quantiles                mode()
skewness                 proportion of each category
```

In [ ]:
# Univariate analysis on numeric columns
# We go beyond .describe() to compute additional insights

numeric_cols = ["study_hours", "attendance", "prev_score",
                "math_score", "science_score", "english_score", "final_score"]

rows = []
for col in numeric_cols:
    s = students[col]
    rows.append({
        "column":   col,
        "mean":     round(s.mean(), 2),
        "median":   round(s.median(), 2),
        "std":      round(s.std(), 2),
        "min":      s.min(),
        "max":      s.max(),
        "range":    round(s.max() - s.min(), 2),
        "skew":     round(s.skew(), 3),   # >0 right-skewed, <0 left-skewed
        "mean_median_gap": round(abs(s.mean() - s.median()), 2)
    })

univariate_numeric = pd.DataFrame(rows).set_index("column")
print("Univariate numeric summary:")
print(univariate_numeric)

Univariate numeric summary:
                mean  median    std   min   max  range   skew  mean_median_gap
column                                                                        
study_hours     5.01    5.05   1.55   1.0   8.5    7.5 -0.189             0.04
attendance     50.64   50.00   2.34  50.0  66.5   16.5  4.362             0.65
prev_score     73.00   72.95  11.68  40.9  99.0   58.1 -0.077             0.05
math_score     57.37   58.35  13.76  30.0  86.1   56.1 -0.190             0.98
science_score  59.18   61.25  14.07  30.0  89.9   59.9 -0.192             2.07
english_score  56.29   55.40  14.96  30.0  95.6   65.6  0.254             0.89
final_score    57.62   58.15  13.10  30.0  88.2   58.2 -0.063             0.53


### Reading the skewness column

**Skewness** tells you whether the distribution is symmetric or lopsided:

```
skew ≈ 0          symmetric — mean and median are close
skew > 0           right-skewed (positive) — a long tail to the right,
                   mean is pulled above the median by a few high values
skew < 0           left-skewed (negative) — a long tail to the left,
                   mean is pulled below the median by a few low values
```

When skew is large and mean ≠ median, **the median is the better measure of 'typical'** — because it is not dragged by the outliers in the tail.

**Note to look for:** does `final_score` appear more skewed than individual subject scores? Why might that be?

In [ ]:
# Univariate analysis on categorical columns

cat_cols = ["gender", "stream", "school_type", "city_tier",
            "parent_edu", "tuition", "grade"]

for col in cat_cols:
    counts = students[col].value_counts()
    pct    = (students[col].value_counts(normalize=True) * 100).round(1)
    summary = pd.DataFrame({"count": counts, "pct_%": pct})
    print(f"── {col} ──")
    print(summary)
    print()

── gender ──
        count  pct_%
gender              
Male      108   54.0
Female     92   46.0

── stream ──
          count  pct_%
stream                
Science      85   42.5
Commerce     76   38.0
Arts         39   19.5

── school_type ──
             count  pct_%
school_type              
Private        103   51.5
Public          97   48.5

── city_tier ──
           count  pct_%
city_tier              
Tier 1        89   44.5
Tier 2        68   34.0
Tier 3        43   21.5

── parent_edu ──
              count  pct_%
parent_edu                
Graduate         76   38.0
Secondary        52   26.0
Postgraduate     52   26.0
Primary          20   10.0

── tuition ──
         count  pct_%
tuition              
No         102   51.0
Yes         98   49.0

── grade ──
       count  pct_%
grade              
C         60   30.0
F         55   27.5
B         45   22.5
A         34   17.0
A+         6    3.0



In [ ]:
# Specific observations worth noting from the categorical scan

print("Key observations from univariate categorical scan:")
print()

# 1. Grade distribution — is it roughly normal or skewed?
print("1. Grade distribution:")
print(students["grade"].value_counts().sort_index())
total = len(students)
f_pct = (students["grade"] == "F").sum() / total * 100
a_pct = ((students["grade"] == "A") | (students["grade"] == "A+")).sum() / total * 100
print(f"   Failing (F): {f_pct:.1f}%   |   A or A+: {a_pct:.1f}%")
print()

# 2. Is the dataset gender-balanced?
print("2. Gender balance:")
print(students["gender"].value_counts())
print()

# 3. How is tuition distributed?
print("3. Tuition prevalence:")
print(students["tuition"].value_counts(normalize=True).round(3) * 100)

Key observations from univariate categorical scan:

1. Grade distribution:
grade
F     55
C     60
B     45
A     34
A+     6
Name: count, dtype: int64
   Failing (F): 27.5%   |   A or A+: 20.0%

2. Gender balance:
gender
Male      108
Female     92
Name: count, dtype: int64

3. Tuition prevalence:
tuition
No     51.0
Yes    49.0
Name: proportion, dtype: float64


---
# Part 3 — Multivariate Analysis

## What it means

**Multivariate** = two or more variables at a time.

Univariate analysis tells you about each column in isolation. Multivariate analysis asks: **do two variables move together?** Does one group perform differently from another? Does knowing the value of one column help predict the value of another?

## Whiteboard: the three relationship types

```
Numeric vs Numeric      -> correlation, cross-tabulation
  "Does study_hours predict final_score?"

Categorical vs Numeric  -> group means (groupby)
  "Do Science students score higher than Arts students?"

Categorical vs Categorical -> cross-tab, chi-square
  "Are Private school students more likely to take tuition?"
```

We look at all three today.

In [ ]:
pd.set_option('display.max_rows', 30)
students

,student_id,gender,stream,school_type,city_tier,parent_edu,study_group,tuition,study_hours,attendance,prev_score,math_score,science_score,english_score,final_score,grade
0,S001,Male,Commerce,Private,Tier 1,Secondary,Group D,No,7.2,50.0,86.7,79.2,86.2,64.1,76.5,A
1,S002,Female,Science,Public,Tier 1,Primary,Group C,No,3.3,50.0,57.5,30.0,30.0,30.0,30.0,F
2,S003,Female,Science,Private,Tier 1,Secondary,Group B,Yes,4.5,50.0,92.1,71.4,82.1,63.6,72.4,A
3,S004,Female,Arts,Public,Tier 1,Graduate,Group A,No,5.8,50.0,77.0,67.8,60.1,69.2,65.7,B
4,S005,Male,Commerce,Private,Tier 1,Graduate,Group D,No,4.8,50.0,63.5,55.7,65.7,63.1,61.5,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,S196,Male,Arts,Private,Tier 3,Secondary,Group B,No,7.2,50.0,72.3,85.2,85.7,92.4,87.8,A+
196,S197,Female,Arts,Public,Tier 3,Secondary,Group C,Yes,4.7,50.0,75.7,51.5,63.2,55.8,56.8,C
197,S198,Female,Science,Private,Tier 2,Secondary,Group B,Yes,6.0,50.0,92.4,85.2,70.5,85.7,80.5,A+
198,S199,Female,Commerce,Private,Tier 2,Postgraduate,Group C,No,6.1,50.0,74.9,56.9,61.4,50.0,56.1,C


In [ ]:
# Categorical vs Numeric: group means
# The workhorse: groupby + mean + comparison

print("=" * 55)
print("Average final_score by STREAM")
print("=" * 55)
stream_scores = students.groupby("stream")["final_score"].agg(
    count="count", mean="mean", median="median", std="std"
).round(2)
print(stream_scores)
print()

print("=" * 55)
print("Average final_score by SCHOOL TYPE")
print("=" * 55)
school_scores = students.groupby("school_type")["final_score"].agg(
    count="count", mean="mean", median="median", std="std"
).round(2)
print(school_scores)
print()

print("=" * 55)
print("Average final_score by TUITION")
print("=" * 55)
tuition_scores = students.groupby("tuition")["final_score"].agg(
    count="count", mean="mean", median="median"
).round(2)
print(tuition_scores)

Average final_score by STREAM
          count   mean  median    std
stream                               
Arts         39  53.08    52.7  13.53
Commerce     76  56.23    56.1  13.38
Science      85  60.94    62.1  11.89

Average final_score by SCHOOL TYPE
             count   mean  median    std
school_type                             
Private        103  61.06    61.0  11.93
Public          97  53.96    54.2  13.35

Average final_score by TUITION
         count   mean  median
tuition                      
No         102  55.70    55.7
Yes         98  59.61    59.7


In [ ]:
# Categorical vs Categorical: cross-tabulation
# pd.crosstab() counts how many rows fall into each combination

print("Cross-tab: school_type vs tuition")
ct = pd.crosstab(students["school_type"], students["tuition"])
print(ct)
print()

# Normalize to show proportions within each row (what % of each school type has tuition)
ct_pct = pd.crosstab(students["school_type"], students["tuition"], normalize="index").round(3) * 100
print("As % of each school type:")
print(ct_pct)
print()

print("Cross-tab: stream vs grade")
print(pd.crosstab(students["stream"], students["grade"]))

Cross-tab: school_type vs tuition
tuition      No  Yes
school_type         
Private      30   73
Public       72   25

As % of each school type:
tuition        No   Yes
school_type            
Private      29.1  70.9
Public       74.2  25.8

Cross-tab: stream vs grade
grade      F   C   B   A  A+
stream                      
Arts      16  13   6   3   1
Commerce  23  26  15  10   2
Science   16  21  24  21   3


In [ ]:
# Slicing with two categorical dimensions
# How does final_score vary across BOTH school_type AND stream?

two_dim = students.groupby(["school_type", "stream"])["final_score"].mean().round(2).unstack()
print("Mean final_score by school_type (rows) and stream (columns):")
print(two_dim)
print()
print("Reading this table: each cell is the average score for that combination.")
print("Look for the highest and lowest cells — those are interesting signals.")

Mean final_score by school_type (rows) and stream (columns):
stream       Arts  Commerce  Science
school_type                         
Private      57.1     59.73    64.93
Public       47.3     51.91    57.71

Reading this table: each cell is the average score for that combination.
Look for the highest and lowest cells — those are interesting signals.


In [ ]:
pd.crosstab(
    students["school_type"],
    students["stream"],
    values=students["final_score"],
    aggfunc="mean").round(2)

stream,Arts,Commerce,Science
school_type,,,
Private,57.1,59.73,64.93
Public,47.3,51.91,57.71


In [ ]:
pd.cut(
    students["study_hours"],
    bins=[0, 3, 5, 7, 11],
    labels=["Low (<3h)", "Medium (3-5h)", "High (5-7h)", "Very High (>7h)"]
)

,study_hours
0,Very High (>7h)
1,Medium (3-5h)
2,Medium (3-5h)
3,High (5-7h)
4,Medium (3-5h)
...,...
195,Very High (>7h)
196,Medium (3-5h)
197,High (5-7h)
198,High (5-7h)


In [ ]:
# Numeric vs Numeric: binning to create a cross-tab
# We cannot cross-tab two continuous variables — we bin one first

students["study_band"] = pd.cut(
    students["study_hours"],
    bins=[0, 3, 5, 7, 11],
    labels=["Low (<3h)", "Medium (3-5h)", "High (5-7h)", "Very High (>7h)"]
)

score_by_study = students.groupby("study_band", observed=True)["final_score"].agg(
    count="count", mean="mean", median="median"
).round(2)

print("Average final_score by study_hours band:")
print(score_by_study)
print()
print("Does more study = higher score? Check the mean column going down.")

Average final_score by study_hours band:
                 count   mean  median
study_band                           
Low (<3h)           26  41.72   42.55
Medium (3-5h)       74  53.96   54.05
High (5-7h)         78  62.30   62.80
Very High (>7h)     22  72.09   72.40

Does more study = higher score? Check the mean column going down.


---
# Part 4 — Correlation Analysis

## What correlation measures

**Correlation** is a number between −1 and +1 that summarises how two numeric variables move together:

```
+1.0    perfect positive — when X goes up, Y always goes up by the same proportion
+0.7    strong positive  — when X goes up, Y tends to go up
+0.3    weak positive    — slight tendency to move together
 0.0    no linear relationship
−0.3    weak negative    — slight tendency to move opposite
−0.7    strong negative  — when X goes up, Y tends to go down
−1.0    perfect negative — when X goes up, Y always goes down by the same proportion
```

## The critical warning: correlation ≠ causation

A correlation between two variables does NOT mean one causes the other. It means they tend to move together — which could be because:
1. A causes B
2. B causes A
3. A third variable C causes both A and B
4. Pure coincidence (especially in small datasets)

EDA finds correlations. Determining causation requires careful experimental design — which is a different (and harder) problem.

In [ ]:
# Full correlation matrix

numeric_cols = ["study_hours", "attendance", "prev_score",
                "math_score", "science_score", "english_score", "final_score"]

corr_matrix = students[numeric_cols].corr().round(3)

print("Correlation matrix (Pearson r):")
corr_matrix

Correlation matrix (Pearson r):


,study_hours,attendance,prev_score,math_score,science_score,english_score,final_score
study_hours,1.000,0.380,0.066,0.641,0.583,0.550,0.642
attendance,0.380,1.000,0.032,0.218,0.214,0.167,0.216
prev_score,0.066,0.032,1.000,0.332,0.346,0.297,0.353
math_score,0.641,0.218,0.332,1.000,0.810,0.774,0.935
science_score,0.583,0.214,0.346,0.810,1.000,0.715,0.914
english_score,0.550,0.167,0.297,0.774,0.715,1.000,0.907
final_score,0.642,0.216,0.353,0.935,0.914,0.907,1.000


In [ ]:
# Reading the correlation matrix systematically
# Extract the correlations with final_score specifically

print("Correlations with final_score (sorted strongest to weakest):")
final_corr = corr_matrix["final_score"].drop("final_score").abs().sort_values(ascending=False)
for col, val in final_corr.items():
    signed = corr_matrix.loc[col, "final_score"]
    strength = "strong" if abs(signed) >= 0.5 else "moderate" if abs(signed) >= 0.3 else "weak"
    direction = "positive" if signed > 0 else "negative"
    print(f"  {col:<20} r = {signed:+.3f}   ({strength} {direction})")

Correlations with final_score (sorted strongest to weakest):
  math_score           r = +0.935   (strong positive)
  science_score        r = +0.914   (strong positive)
  english_score        r = +0.907   (strong positive)
  study_hours          r = +0.642   (strong positive)
  prev_score           r = +0.353   (moderate positive)
  attendance           r = +0.216   (weak positive)


In [ ]:
# Extract ALL pairs above a correlation threshold
# Useful when the matrix is large and you want to focus on the strongest signals

threshold = 0.3

# Get the upper triangle only (avoid counting each pair twice)
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Stack into a Series, filter by threshold
strong_corr = (
    upper_triangle.stack()
    .reset_index()
    .rename(columns={"level_0": "var1", "level_1": "var2", 0: "r"})
)
strong_corr = strong_corr[strong_corr["r"].abs() >= threshold].sort_values("r", ascending=False)

print(f"Variable pairs with |r| >= {threshold}:")
print(strong_corr.to_string(index=False))

Variable pairs with |r| >= 0.3:
         var1          var2     r
   math_score   final_score 0.935
science_score   final_score 0.914
english_score   final_score 0.907
   math_score science_score 0.810
   math_score english_score 0.774
science_score english_score 0.715
  study_hours   final_score 0.642
  study_hours    math_score 0.641
  study_hours science_score 0.583
  study_hours english_score 0.550
  study_hours    attendance 0.380
   prev_score   final_score 0.353
   prev_score science_score 0.346
   prev_score    math_score 0.332


In [ ]:
# Is the study_hours <-> final_score correlation consistent across streams?
# A correlation can look strong overall but vary dramatically between subgroups

print("Correlation between study_hours and final_score by stream:")
for stream_name, group in students.groupby("stream"):
    r = group["study_hours"].corr(group["final_score"])
    print(f"  {stream_name:<12}  r = {r:.3f}")
print()
print("If the correlation varies between groups, the overall r is misleading.")
print("This is called Simpson's Paradox when a trend reverses across subgroups.")

Correlation between study_hours and final_score by stream:
  Arts          r = 0.674
  Commerce      r = 0.744
  Science       r = 0.556

If the correlation varies between groups, the overall r is misleading.
This is called Simpson's Paradox when a trend reverses across subgroups.
